# 11 - Advanced Analysis: Complete Paper-Worthy Pipeline

This notebook runs the entire upgraded analysis pipeline:
1. Data Loading & Preprocessing
2. Exploratory Data Analysis
3. Clustering (KMeans, DBSCAN, Hierarchical, GMM)
4. Classification (8 models with k-fold CV)
5. Regression (7 models with proper metrics)
6. Association Rules (Fixed Apriori)
7. Advanced Analysis (t-SNE, Outlier Detection, Statistical Tests)
8. Comprehensive Summary Report


## Setup & Imports


In [ ]:
import sys
import os
import warnings

warnings.filterwarnings('ignore')

# Add project root to path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Import our modules
from src.preprocessing import (
    load_data, clean_data, encode_gender, normalize_data,
    add_age_bins, add_income_category, add_spending_category
)
from src.eda import (
    plot_age_distribution, plot_correlation, plot_income_vs_spending,
    plot_gender_distribution, plot_distributions, plot_boxplots
)
from src.clustering import (
    elbow_method, compare_clustering, plot_all_clusters,
    plot_dendrogram, apply_kmeans
)
from src.models import (
    prepare_classification_data, compare_classifiers,
    plot_feature_importance, plot_model_comparison,
    plot_confusion_matrices,
    tune_random_forest, tune_gradient_boosting, tune_knn,
    cross_validate_model, get_all_models
)
from src.regression import (
    prepare_regression_data, compare_regressors,
    plot_regression_comparison, plot_all_regressions_1d, plot_regression
)
from src.apriori import (
    prepare_data, prepare_data_multibin, apply_apriori, print_top_rules
)
from src.advanced_analysis import (
    plot_pca_vs_tsne, detect_outliers_isolation_forest,
    detect_outliers_lof, plot_outliers, compare_all_pairs,
    generate_summary_report
)

# Ensure output directories exist
os.makedirs('outputs/plots', exist_ok=True)
os.makedirs('outputs/results', exist_ok=True)
os.makedirs('outputs/reports', exist_ok=True)

print("Setup complete!")

---
## Step 1: Data Loading & Preprocessing


In [ ]:
df = load_data('data/raw/mall_customers.csv')
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
print(f"Data types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
df.describe()

In [ ]:
# Clean data
df = clean_data(df)
print(f"After cleaning: {df.shape}")

# Store original values BEFORE normalization for Apriori and display
df_original = df.copy()

# Add categorical columns BEFORE normalization
df_original = add_age_bins(df_original)
df_original = add_income_category(df_original)
df_original = add_spending_category(df_original)

# Encode gender
df = encode_gender(df)
print(f"Gender encoded. Unique values: {df['Gender'].unique()}")

# Normalize
df_normalized, scaler = normalize_data(df, method='minmax')
print(f"Data normalized (MinMax). Shape: {df_normalized.shape}")

---
## Step 2: Exploratory Data Analysis


In [ ]:
plot_distributions(df_original, save_path='outputs/plots/feature_distributions.png')

In [ ]:
plot_age_distribution(df_original, save_path='outputs/plots/age_distribution.png')

In [ ]:
plot_income_vs_spending(df_original, save_path='outputs/plots/income_vs_spending.png')

In [ ]:
plot_gender_distribution(df_original, save_path='outputs/plots/gender_distribution.png')

In [ ]:
plot_correlation(df_normalized, save_path='outputs/plots/correlation_matrix.png')

In [ ]:
plot_boxplots(df_original, save_path='outputs/plots/boxplots.png')

---
## Step 3: Clustering Analysis

Comparing 4 clustering methods: **KMeans**, **DBSCAN**, **Agglomerative Hierarchical**, and **Gaussian Mixture Model (GMM)**.

Evaluation metrics: **Silhouette Score**, **Davies-Bouldin Index**, **Calinski-Harabasz Index**.


In [ ]:
# Prepare features for clustering (2D: Income + Spending)
X_cluster_2d = df_normalized[['Annual Income (k$)', 'Spending Score (1-100)']].values

# Elbow Method + Silhouette
wcss, sil_scores = elbow_method(X_cluster_2d)

In [ ]:
# Hierarchical Dendrogram
plot_dendrogram(X_cluster_2d)

In [ ]:
# Compare all clustering methods (2D)
cluster_results_df, all_labels = compare_clustering(
    X_cluster_2d, n_clusters=5, dbscan_eps=0.15, dbscan_min_samples=5
)
cluster_results_df.to_csv('outputs/results/clustering_comparison.csv', index=False)
cluster_results_df

In [ ]:
# Plot all clustering results side by side
plot_all_clusters(X_cluster_2d, all_labels, xlabel="Income", ylabel="Spending")

### Multi-Feature Clustering (4 features)


In [ ]:
X_cluster_4d = df_normalized[['Age', 'Gender', 'Annual Income (k$)',
                               'Spending Score (1-100)']].values
cluster_results_4d, all_labels_4d = compare_clustering(
    X_cluster_4d, n_clusters=5, dbscan_eps=0.4, dbscan_min_samples=5
)

In [ ]:
# Add cluster labels to dataframe using KMeans (best default)
df_normalized['Cluster'] = all_labels['KMeans']
df_original['Cluster'] = all_labels['KMeans']

print("Cluster distribution:")
df_normalized['Cluster'].value_counts().sort_index()

---
## Step 4: Classification

Training **8 classifiers**: Decision Tree, SVM, Random Forest, KNN, Gradient Boosting, Naive Bayes, Logistic Regression, MLP Neural Network.

Evaluation: **Accuracy, Precision, Recall, F1-Score, ROC-AUC** + **10-Fold Cross-Validation**.


In [ ]:
# Prepare data (uses all features to predict cluster label)
X_train, X_test, y_train, y_test, feature_names = prepare_classification_data(
    df_normalized, target_col='Cluster'
)
print(f"Features: {feature_names}")
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
print(f"Classes: {sorted(y_train.unique())}")

In [ ]:
# Compare all classifiers
X_full = pd.concat([X_train, X_test])
y_full = pd.concat([y_train, y_test])

clf_results_df, cv_results_df, trained_models = compare_classifiers(
    X_train, X_test, y_train, y_test, X_full, y_full, cv=10
)

# Save results
clf_results_df.to_csv('outputs/results/classification_comparison.csv', index=False)
cv_results_df.to_csv('outputs/results/classification_cv_results.csv', index=False)

In [ ]:
# Display results as tables
print("\n=== Test Set Results ===")
display(clf_results_df)
print("\n=== Cross-Validation Results ===")
display(cv_results_df)

In [ ]:
# Model comparison charts
plot_model_comparison(clf_results_df, metric='F1-Score',
                       save_path='outputs/plots/classification_f1_comparison.png')

In [ ]:
plot_model_comparison(clf_results_df, metric='Accuracy',
                       save_path='outputs/plots/classification_accuracy_comparison.png')

In [ ]:
# Confusion matrices for all models
plot_confusion_matrices(trained_models, X_test, y_test,
                         save_path='outputs/plots/confusion_matrices.png')

### Feature Importance


In [ ]:
if 'Random Forest' in trained_models:
    plot_feature_importance(
        trained_models['Random Forest'], feature_names, 'Random Forest',
        save_path='outputs/plots/feature_importance_rf.png'
    )

In [ ]:
if 'Gradient Boosting' in trained_models:
    plot_feature_importance(
        trained_models['Gradient Boosting'], feature_names, 'Gradient Boosting',
        save_path='outputs/plots/feature_importance_gb.png'
    )

### Hyperparameter Tuning (GridSearchCV)


In [ ]:
best_rf, rf_params, rf_score = tune_random_forest(X_train, y_train)
best_gb, gb_params, gb_score = tune_gradient_boosting(X_train, y_train)
best_knn, knn_params, knn_score = tune_knn(X_train, y_train)

# Save tuning results
tuning_results = pd.DataFrame([
    {'Model': 'Random Forest', 'Best_Params': str(rf_params), 'Best_CV_F1': rf_score},
    {'Model': 'Gradient Boosting', 'Best_Params': str(gb_params), 'Best_CV_F1': gb_score},
    {'Model': 'KNN', 'Best_Params': str(knn_params), 'Best_CV_F1': knn_score},
])
tuning_results.to_csv('outputs/results/hyperparameter_tuning.csv', index=False)
display(tuning_results)

---
## Step 5: Regression Analysis

Training **7 regression models**: Linear, Polynomial (deg=3), Ridge, Lasso, Decision Tree, Random Forest, SVR.

Evaluation: **R-squared, Adjusted R-squared, MSE, RMSE, MAE** + **Cross-Validation R-squared**.


### Single Feature: Income -> Spending


In [ ]:
X_reg_train, X_reg_test, y_reg_train, y_reg_test, reg_features = \
    prepare_regression_data(df_normalized, feature_cols=['Annual Income (k$)'])

reg_results_df, reg_trained = compare_regressors(
    X_reg_train, X_reg_test, y_reg_train, y_reg_test, poly_degree=3
)
reg_results_df.to_csv('outputs/results/regression_comparison.csv', index=False)
display(reg_results_df)

In [ ]:
# R-squared comparison chart
plot_regression_comparison(reg_results_df,
                            save_path='outputs/plots/regression_r2_comparison.png')

In [ ]:
# All regression lines on one plot
plot_all_regressions_1d(reg_trained, X_reg_train, X_reg_test,
                         y_reg_train, y_reg_test,
                         save_path='outputs/plots/all_regression_lines.png')

In [ ]:
# Best model: actual vs predicted + residuals
best_reg_name = reg_results_df.iloc[0]['Model']
plot_regression(reg_trained[best_reg_name], X_reg_train, X_reg_test,
                y_reg_train, y_reg_test, best_reg_name,
                save_path='outputs/plots/best_regression_detail.png')

### Multi-Feature: Age + Gender + Income -> Spending


In [ ]:
X_reg_train_m, X_reg_test_m, y_reg_train_m, y_reg_test_m, reg_features_m = \
    prepare_regression_data(
        df_normalized,
        feature_cols=['Age', 'Gender', 'Annual Income (k$)']
    )

reg_results_multi, _ = compare_regressors(
    X_reg_train_m, X_reg_test_m, y_reg_train_m, y_reg_test_m, poly_degree=2
)
reg_results_multi.to_csv('outputs/results/regression_multi_feature.csv', index=False)
display(reg_results_multi)

---
## Step 6: Association Rule Mining (Fixed Apriori)

**Bug fix**: Uses **median-based** and **quantile-based** categorical splits instead of the previous hardcoded 0.5 threshold that produced misleading 98-99% confidence rules.


### Binary Rules (Median Split)


In [ ]:
basket_binary = prepare_data(df_original)
freq_items_bin, rules_binary = apply_apriori(basket_binary,
                                              min_support=0.15,
                                              min_confidence=0.5)
print_top_rules(rules_binary, n=10)

### Multi-Bin Rules (Tercile Split)


In [ ]:
basket_multi = prepare_data_multibin(df_original)
freq_items_multi, rules_multi = apply_apriori(basket_multi,
                                               min_support=0.1,
                                               min_confidence=0.4)
print_top_rules(rules_multi, n=10)

In [ ]:
# Save rules to CSV
if len(rules_binary) > 0:
    rules_save = rules_binary.copy()
    rules_save['antecedents'] = rules_save['antecedents'].apply(
        lambda x: ', '.join(list(x)))
    rules_save['consequents'] = rules_save['consequents'].apply(
        lambda x: ', '.join(list(x)))
    rules_save.to_csv('outputs/results/association_rules_binary.csv', index=False)

if len(rules_multi) > 0:
    rules_save_m = rules_multi.copy()
    rules_save_m['antecedents'] = rules_save_m['antecedents'].apply(
        lambda x: ', '.join(list(x)))
    rules_save_m['consequents'] = rules_save_m['consequents'].apply(
        lambda x: ', '.join(list(x)))
    rules_save_m.to_csv('outputs/results/association_rules_multibin.csv', index=False)

print("Rules saved!")

---
## Step 7: Advanced Analysis

- **PCA vs t-SNE** dimensionality reduction comparison
- **Outlier Detection** with Isolation Forest + Local Outlier Factor
- **Statistical Significance Testing** with paired t-tests


### PCA vs t-SNE Comparison


In [ ]:
X_all = df_normalized[['Age', 'Gender', 'Annual Income (k$)',
                         'Spending Score (1-100)']].values
pca_result, tsne_result = plot_pca_vs_tsne(
    X_all, labels=df_normalized['Cluster'].values,
    save_path='outputs/plots/pca_vs_tsne.png'
)

### Outlier Detection


In [ ]:
outliers_iso, scores_iso = detect_outliers_isolation_forest(X_cluster_2d)
outliers_lof, scores_lof = detect_outliers_lof(X_cluster_2d)

plot_outliers(X_cluster_2d, outliers_iso, outliers_lof,
              xlabel="Income", ylabel="Spending",
              save_path='outputs/plots/outlier_detection.png')

# Show overlap
both_outliers = outliers_iso & outliers_lof
print(f"\nOutliers detected by BOTH methods: {both_outliers.sum()}")
print(f"Only Isolation Forest: {(outliers_iso & ~outliers_lof).sum()}")
print(f"Only LOF: {(outliers_lof & ~outliers_iso).sum()}")

### Statistical Significance Tests (Paired t-test)


In [ ]:
import sklearn.base
models_dict = get_all_models()
cv_raw_results = []
for name, model in models_dict.items():
    fresh = sklearn.base.clone(model)
    cv_res = cross_validate_model(fresh, X_full, y_full, cv=10, model_name=name)
    cv_raw_results.append(cv_res)

p_value_matrix = compare_all_pairs(cv_raw_results)
p_value_matrix.to_csv('outputs/results/pairwise_pvalues.csv')

---
## Step 8: Comprehensive Summary


In [ ]:
report = generate_summary_report(
    clf_results=clf_results_df,
    reg_results=reg_results_df,
    cluster_results=cluster_results_df,
    save_path='outputs/reports/summary_report.txt'
)

print("\nAll done! All plots saved to outputs/plots/")
print("All results saved to outputs/results/")
print("Summary report saved to outputs/reports/summary_report.txt")